Yeh example [TensorFlow ke original tutorial](https://www.tensorflow.org/tutorials/generative/style_transfer) se inspired hai, saath hi [is blog post](https://medium.com/mlreview/making-ai-art-with-style-transfer-using-keras-8bb5fa44b216) se bhi. CNTK framework use karke Style Transfer ka ek aur achha example [yahan](https://github.com/Microsoft/CNTK/blob/master/Tutorials/CNTK_205_Artistic_Style_Transfer.ipynb) available hai. [Artistic Style Transfer](https://arxiv.org/abs/1508.06576) par original research paper yahan hai.

Style Transfer ke peeche ke main ideas yeh hain:

* White noise se shuru karke, hum current image $x$ ko optimize karne ki koshish karte hain taaki koi loss function minimize ho jaye.
* Loss function teen components se milkar bana hota hai $\mathcal{L(x)} = \alpha\mathcal{L}_c(x,i) + \beta\mathcal{L}_s(x,s)+\gamma\mathcal{L}_t(x)$
   - $\mathcal{L}_c$ - content loss - yeh dikhata hai ki current image $x$ original image $i$ ke kitni kareeb hai.
   - $\mathcal{L}_s$ - style loss - yeh dikhata hai ki current image $x$ style image $s$ ke kitni kareeb hai.
   - $\mathcal{L}_t$ - total variation loss (ise hum apne example mein consider nahi karenge) - yeh ensure karta hai ki resulting image smooth ho, yaani yeh image $x$ ke neighboring pixels ki average squared error ko dikhata hai.

In loss functions ko samajhdaari se design karna padta hai, taaki jaise style loss images ki style similarity ko capture kare, na ki actual content ko. Iske liye hum ek CNN ke kuch deep feature layers ko compare karenge, jo image ko dekhta hai.

Chalo kuch images load karke shuru karte hain:

In [ ]:
!mkdir -p images
!curl https://cdn.pixabay.com/photo/2016/05/18/00/27/franz-marc-1399594_960_720.jpg > images/style.jpg
!curl https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_tabby_and_white_kitten_n01.jpg/1280px-Golden_tabby_and_white_kitten_n01.jpg > images/image.jpg

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input
import IPython.display as display

In [ ]:
img_size = 256

def load_image(fn):
  x = cv2.imread(fn)
  return cv2.cvtColor(x, cv2.COLOR_BGR2RGB)

img_style = load_image('images/style.jpg')
img_content = load_image('images/image.jpg')
img_content = img_content[:,200:200+857,:]
img_content = cv2.resize(img_content,(img_size,img_size))
img_style = img_style[:,200:200+671,:]
img_style = cv2.resize(img_style,(img_size,img_size))

img_result = np.random.uniform(size=(img_size,img_size,3))

matplotlib.rcParams['figure.figsize'] = (12, 12)
matplotlib.rcParams['axes.grid'] = False

fig,ax = plt.subplots(1,3)
ax[0].imshow(img_content)
ax[1].imshow(img_style)
ax[2].imshow((255*img_result).astype(int))
plt.show()

Style loss aur content loss calculate karne ke liye, hamein ek CNN dwara nikale gaye feature space mein kaam karna hoga. Hum alag-alag CNN architectures use kar sakte hain, lekin hamare case mein simplicity ke liye hum *ImageNet* par pre-trained VGG-19 choose karenge.

In [ ]:
vgg = tf.keras.applications.VGG16(include_top=False, weights='imagenet')
vgg.trainable = False

In [ ]:
vgg.summary()

Aaiye ek function define karte hain jo hamein VGG network se intermediate features extract karne dega:

In [ ]:
def layer_extractor(layers):
    outputs = [vgg.get_layer(x).output for x in layers]
    model = tf.keras.Model([vgg.input],outputs)
    return model 

## Content Loss

**Content loss** dikhayega ki hamari current image $x$ original image ke kitni kareeb hai. Yeh CNN ke intermediate feature layers ko dekhta hai aur squared error calculate karta hai. Layer $l$ par content loss ko is tarah define kiya jaayega:
$$
\mathcal{L}_c = {1\over2}\sum_{i,j} (F_{ij}^{(l)}-P_{ij}^{(l)})^2
$$
jahan $F^{(l)}$ aur $P^{(l)}$ -- layer $l$ par features hain.

In [ ]:
content_layers = ['block4_conv2'] 
content_extractor = layer_extractor(content_layers)

content_target = content_extractor(preprocess_input(tf.expand_dims(img_content,axis=0)))

def content_loss(img):
    z = content_extractor(preprocess_input(tf.expand_dims(255*img,axis=0))) 
    return 0.5*tf.reduce_sum((z-content_target)**2)

Ab hum style transfer ki main trick - **optimization** - implement karenge. Hum ek random image se shuru karenge, aur phir content loss ko minimize karne ke liye is image ko adjust karne ke liye TensorFlow optimizer use karenge.

**Important**: Hamare case mein, saari calculations GPU-enabled *TensorFlow* framework se ki jaati hain, jo is code ko GPU par bahut zyada efficiently chalne deta hai.

In [ ]:
img = tf.Variable(img_result)
opt = tf.optimizers.Adam(learning_rate=0.002, beta_1=0.99, epsilon=1e-1)

clip = lambda x : tf.clip_by_value(x,clip_value_min=0,clip_value_max=1)

def optimize(img,loss_fn):
    with tf.GradientTape() as tape:
        loss = loss_fn(img)
    grad = tape.gradient(loss,img)
    opt.apply_gradients([(grad,img)])
    #img.assign(tf.clip_by_value(img,clip_value_min=0,clip_value_max=1))

def train(img,loss_fn,epochs=10,steps_per_epoch=100):
    for _ in range(epochs):
        display.clear_output(wait=True)
        plt.imshow((255*clip(img)).numpy().astype(int))
        plt.show()
        for _ in range(steps_per_epoch):
            optimize(img,loss_fn=loss_fn)

train(img,content_loss)

> **Exercise**: Network ki alag-alag layers ke saath experiment karke dekhiye ki kya hota hai. Aap kai layers ko ek saath optimize karne ki bhi koshish kar sakte hain, lekin iske liye aapko `content_loss` ke code mein thoda change karna padega.

## Style Loss

Style loss, style transfer ke peeche ka main idea hai. Hum actual features ko compare nahi karte, balki unke Gram matrices ko compare karte hain, jinhe is tarah define kiya jaata hai: $$G=A\times A^T$$

Gram matrix, correlation matrix jaisa hota hai aur yeh dikhata hai ki kuch filters doosre filters par kaise depend karte hain. Style loss ko alag-alag layers se aane wale losses ke sum ke roop mein calculate kiya jaata hai, jinhe aksar weighted coefficients ke saath consider kiya jaata hai.

Style transfer ke liye total loss function *content loss* aur *style loss* ka sum hota hai.

In [ ]:
def gram_matrix(x):
  result = tf.linalg.einsum('bijc,bijd->bcd', x, x)
  input_shape = tf.shape(x)
  num_locations = tf.cast(input_shape[1]*input_shape[2], tf.float32)
  return result/(num_locations)

style_layers = ['block1_conv1','block2_conv1','block3_conv1','block4_conv1']

def style_extractor(img):
    return [gram_matrix(x) for x in layer_extractor(style_layers)(img)]

style_target = style_extractor(preprocess_input(tf.expand_dims(img_style,axis=0)))

def style_loss(img):
    z = style_extractor(preprocess_input(tf.expand_dims(255*img,axis=0)))
    loss = tf.add_n([tf.reduce_mean((x-target)**2) 
                           for x,target in zip(z,style_target)])
    return loss / len(style_layers)


## Sab Kuch Ek Saath Jodna

Hum `total_loss` function define karenge jo combined loss calculate karega, aur optimization process chalayega:

In [ ]:
def total_loss(img):
    return 2*content_loss(img)+style_loss(img)

img.assign(img_result)

train(img,loss_fn=total_loss)

Neeche diya gaya code loss ka actual optimization karta hai. Dhyan rakhiye ki GPU ke saath bhi optimization mein kaafi time lagta hai. Results behtar karne ke liye aap neeche wali cell ko kai baar run kar sakte hain.

## Variation Loss Add Karna

**Variation loss** hamein image ko kam noisy banane mein help karta hai, neighboring pixels ke beech ka difference kam karke.

Hum optimization ko original content image se shuru karenge, jisse hum image mein zyada content details bacha kar rakh sakte hain, bina content loss function ko complicated kiye. Haalaanki, hum thoda noise add karenge.

In [ ]:
def variation_loss(img):
  img = tf.cast(img,tf.float32)
  x_var = img[ :, 1:, :] - img[ :, :-1, :]
  y_var = img[ 1:, :, :] - img[ :-1, :, :]
  return tf.reduce_sum(tf.abs(x_var)) + tf.reduce_sum(tf.abs(y_var))
  
def total_loss_var(img):
    return content_loss(img)+150*style_loss(img)+30*variation_loss(img)

img.assign(clip(np.random.normal(-0.3,0.3,size=img_content.shape)+img_content/255.0))

train(img,loss_fn=total_loss_var)

In [ ]:
cv2.imwrite('result.jpg',(img.numpy()[:,:,::-1]*255))

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->